Integrar best embedding al LLM amb el chunk size i overlap adient.

intfloat/multilingual-e5-large

In [ ]:
# Importar data
from google.colab import files
import pandas as pd

uploaded = files.upload()

Saving chunks_df.csv to chunks_df.csv
Saving qa_df.csv to qa_df.csv


In [ ]:
chunks_df = pd.read_csv("chunks_df.csv")
qa_df = pd.read_csv("qa_df.csv")

    doc_id            chunk_id  posicio  \
0  DOC_001  DOC_001_chunk_0000        0   
1  DOC_001  DOC_001_chunk_0001        1   
2  DOC_001  DOC_001_chunk_0002        2   
3  DOC_001  DOC_001_chunk_0003        3   
4  DOC_001  DOC_001_chunk_0004        4   

                                          text_chunk  \
0      Regulations for External Academic Internships   
1  The statistics degree is from the University o...   
2  . The program has a total of 240 credits. Requ...   
3  . e) At the time the intership begins, the und...   
4  Duration and Schedules of Internships  The ext...   

                                     text_chunk_norm  
0      regulations for external academic internships  
1  the statistics degree is from the university o...  
2  the program has a total of 240 credits require...  
3  e at the time the intership begins the undergr...  
4  duration and schedules of internships the exte...  


In [ ]:
# e5 and flan xl 9 horas i 24 minutos
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import torch

# Model d'embedding (recuperador) - multilingual-e5-large amb prefixos
modelo_emb = SentenceTransformer('intfloat/multilingual-e5-large')

# Model de llenguatge (generador) - FLAN-T5-XL
model_name = "google/flan-t5-xl"
tokenizer_llm = AutoTokenizer.from_pretrained(model_name)
modelo_llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo_llm = modelo_llm.to(device)

print(f"✅ Models carregats a {device}")

# -----------------------------------
# 2. Precalcular embeddings dels chunks (NOMÉS UN COP)
#    Atenció: afegir el prefix "passage: "
# -----------------------------------
print("Precalculant embeddings dels chunks amb prefix 'passage:'...")
chunks_textos = chunks_df['text_chunk'].tolist()
chunks_textos_prefijo = ["passage: " + texto for texto in chunks_textos]  # IMPORTANT
chunks_ids = chunks_df['chunk_id'].tolist()
chunks_docs = chunks_df['doc_id'].tolist()
chunks_emb = modelo_emb.encode(chunks_textos_prefijo, show_progress_bar=True)
print(f"✅ {len(chunks_emb)} embeddings de chunks calculats")

# -----------------------------------
# 3. Per a cada pregunta: recuperar chunks i generar respostes
# -----------------------------------
top_k = 3

# Llistes per guardar resultats
respuestas_con_rag = []
respuestas_sin_rag = []

# Columnes per als chunks recuperats
chunk_rec_1_id = []
chunk_rec_1_doc = []
chunk_rec_1_texto = []
chunk_rec_1_sim = []

chunk_rec_2_id = []
chunk_rec_2_doc = []
chunk_rec_2_texto = []
chunk_rec_2_sim = []

chunk_rec_3_id = []
chunk_rec_3_doc = []
chunk_rec_3_texto = []
chunk_rec_3_sim = []

print("\nProcessant preguntes...")
for i, (_, row) in enumerate(qa_df.iterrows()):
    pregunta = row['Q']
    short_answer = row['SA']
    long_answer = row['LA']

    # --- RECUPERACIÓ amb el prefix "query: " ---
    pregunta_prefijo = "query: " + pregunta
    pregunta_emb = modelo_emb.encode([pregunta_prefijo], show_progress_bar=False)
    sims = cosine_similarity(pregunta_emb, chunks_emb)[0]
    top_indices = np.argsort(sims)[-top_k:][::-1]

    # Guardar chunks recuperats
    for pos, idx in enumerate(top_indices):
        if pos == 0:
            chunk_rec_1_id.append(chunks_ids[idx])
            chunk_rec_1_doc.append(chunks_docs[idx])
            chunk_rec_1_texto.append(chunks_textos[idx])
            chunk_rec_1_sim.append(float(sims[idx]))
        elif pos == 1:
            chunk_rec_2_id.append(chunks_ids[idx])
            chunk_rec_2_doc.append(chunks_docs[idx])
            chunk_rec_2_texto.append(chunks_textos[idx])
            chunk_rec_2_sim.append(float(sims[idx]))
        elif pos == 2:
            chunk_rec_3_id.append(chunks_ids[idx])
            chunk_rec_3_doc.append(chunks_docs[idx])
            chunk_rec_3_texto.append(chunks_textos[idx])
            chunk_rec_3_sim.append(float(sims[idx]))

    # Textos dels chunks recuperats (sense prefix per al prompt)
    chunks_texto_top = [chunks_textos[idx] for idx in top_indices]
    contexto = "\n\n".join(chunks_texto_top)

    # --- GENERACIÓ AMB RAG ---
    prompt_con_rag = f"""You are a Statistics student at the Faculty of Economics and Business of the University of Barcelona (UB). Use ONLY the following information to answer the question concisely and accurately. If the information is not in the provided documents, say so.

Documents:
{contexto}

Question: {pregunta}
Answer:"""

    inputs = tokenizer_llm(prompt_con_rag, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = modelo_llm.generate(**inputs, max_new_tokens=150, do_sample=False)
    resp_rag = tokenizer_llm.decode(outputs[0], skip_special_tokens=True)

    # --- GENERACIÓ SENSE RAG ---
    prompt_sin_rag = f"""You are a Statistics student at the Faculty of Economics and Business of the University of Barcelona (UB). Answer the following question concisely and accurately. If you don't know the answer, say so.

Question: {pregunta}
Answer:"""

    inputs = tokenizer_llm(prompt_sin_rag, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = modelo_llm.generate(**inputs, max_new_tokens=150, do_sample=False)
    resp_sin_rag = tokenizer_llm.decode(outputs[0], skip_special_tokens=True)

    # Guardar respostes
    respuestas_con_rag.append(resp_rag)
    respuestas_sin_rag.append(resp_sin_rag)

    print(f"  {i+1}/{len(qa_df)}: {pregunta[:80]}...")
    print(f"    SA real: {short_answer}")
    print(f"    LA real: {long_answer}")
    print(f"    Amb RAG:  {resp_rag}")
    print(f"    Sense RAG: {resp_sin_rag}")
    print("    ---")

# -----------------------------------
# 4. Guardar tot al DataFrame
# -----------------------------------
qa_df['respuesta_con_rag'] = respuestas_con_rag
qa_df['respuesta_sin_rag'] = respuestas_sin_rag

qa_df['chunk_rec_1_id'] = chunk_rec_1_id
qa_df['chunk_rec_1_doc'] = chunk_rec_1_doc
qa_df['chunk_rec_1_texto'] = chunk_rec_1_texto
qa_df['chunk_rec_1_sim'] = chunk_rec_1_sim

qa_df['chunk_rec_2_id'] = chunk_rec_2_id
qa_df['chunk_rec_2_doc'] = chunk_rec_2_doc
qa_df['chunk_rec_2_texto'] = chunk_rec_2_texto
qa_df['chunk_rec_2_sim'] = chunk_rec_2_sim

qa_df['chunk_rec_3_id'] = chunk_rec_3_id
qa_df['chunk_rec_3_doc'] = chunk_rec_3_doc
qa_df['chunk_rec_3_texto'] = chunk_rec_3_texto
qa_df['chunk_rec_3_sim'] = chunk_rec_3_sim


print("\n✅ Totes les dades guardades al DataFrame!")
print(f"Columnes del DataFrame: {qa_df.columns.tolist()}")

qa_df.to_csv("qa_df_resultados_flanxl.csv", index=False)
files.download("qa_df_resultados_flanxl.csv")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
# e5 and flan large 21 minuts
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import torch

# Model d'embedding (recuperador) - multilingual-e5-large amb prefixos
modelo_emb = SentenceTransformer('intfloat/multilingual-e5-large')

# Model de llenguatge (generador) - FLAN-T5-XL
model_name = "google/flan-t5-large"
tokenizer_llm = AutoTokenizer.from_pretrained(model_name)
modelo_llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo_llm = modelo_llm.to(device)

print(f"✅ Models carregats a {device}")

# -----------------------------------
# 2. Precalcular embeddings dels chunks (NOMÉS UN COP)
#    Atenció: afegir el prefix "passage: "
# -----------------------------------
print("Precalculant embeddings dels chunks amb prefix 'passage:'...")
chunks_textos = chunks_df['text_chunk'].tolist()
chunks_textos_prefijo = ["passage: " + texto for texto in chunks_textos]  # IMPORTANT
chunks_ids = chunks_df['chunk_id'].tolist()
chunks_docs = chunks_df['doc_id'].tolist()
chunks_emb = modelo_emb.encode(chunks_textos_prefijo, show_progress_bar=True)
print(f"✅ {len(chunks_emb)} embeddings de chunks calculats")

# -----------------------------------
# 3. Per a cada pregunta: recuperar chunks i generar respostes
# -----------------------------------
top_k = 3

# Llistes per guardar resultats
respuestas_con_rag = []
respuestas_sin_rag = []

# Columnes per als chunks recuperats
chunk_rec_1_id = []
chunk_rec_1_doc = []
chunk_rec_1_texto = []
chunk_rec_1_sim = []

chunk_rec_2_id = []
chunk_rec_2_doc = []
chunk_rec_2_texto = []
chunk_rec_2_sim = []

chunk_rec_3_id = []
chunk_rec_3_doc = []
chunk_rec_3_texto = []
chunk_rec_3_sim = []

print("\nProcessant preguntes...")
for i, (_, row) in enumerate(qa_df.iterrows()):
    pregunta = row['Q']
    short_answer = row['SA']
    long_answer = row['LA']

    # --- RECUPERACIÓ amb el prefix "query: " ---
    pregunta_prefijo = "query: " + pregunta
    pregunta_emb = modelo_emb.encode([pregunta_prefijo], show_progress_bar=False)
    sims = cosine_similarity(pregunta_emb, chunks_emb)[0]
    top_indices = np.argsort(sims)[-top_k:][::-1]

    # Guardar chunks recuperats
    for pos, idx in enumerate(top_indices):
        if pos == 0:
            chunk_rec_1_id.append(chunks_ids[idx])
            chunk_rec_1_doc.append(chunks_docs[idx])
            chunk_rec_1_texto.append(chunks_textos[idx])
            chunk_rec_1_sim.append(float(sims[idx]))
        elif pos == 1:
            chunk_rec_2_id.append(chunks_ids[idx])
            chunk_rec_2_doc.append(chunks_docs[idx])
            chunk_rec_2_texto.append(chunks_textos[idx])
            chunk_rec_2_sim.append(float(sims[idx]))
        elif pos == 2:
            chunk_rec_3_id.append(chunks_ids[idx])
            chunk_rec_3_doc.append(chunks_docs[idx])
            chunk_rec_3_texto.append(chunks_textos[idx])
            chunk_rec_3_sim.append(float(sims[idx]))

    # Textos dels chunks recuperats (sense prefix per al prompt)
    chunks_texto_top = [chunks_textos[idx] for idx in top_indices]
    contexto = "\n\n".join(chunks_texto_top)

    # --- GENERACIÓ AMB RAG ---
    prompt_con_rag = f"""You are a Statistics student at the Faculty of Economics and Business of the University of Barcelona (UB). Use ONLY the following information to answer the question concisely and accurately. If the information is not in the provided documents, say so.

Documents:
{contexto}

Question: {pregunta}
Answer:"""

    inputs = tokenizer_llm(prompt_con_rag, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = modelo_llm.generate(**inputs, max_new_tokens=150, do_sample=False)
    resp_rag = tokenizer_llm.decode(outputs[0], skip_special_tokens=True)

    # --- GENERACIÓ SENSE RAG ---
    prompt_sin_rag = f"""You are a Statistics student at the Faculty of Economics and Business of the University of Barcelona (UB). Answer the following question concisely and accurately. If you don't know the answer, say so.

Question: {pregunta}
Answer:"""

    inputs = tokenizer_llm(prompt_sin_rag, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = modelo_llm.generate(**inputs, max_new_tokens=150, do_sample=False)
    resp_sin_rag = tokenizer_llm.decode(outputs[0], skip_special_tokens=True)

    # Guardar respostes
    respuestas_con_rag.append(resp_rag)
    respuestas_sin_rag.append(resp_sin_rag)

    print(f"  {i+1}/{len(qa_df)}: {pregunta[:80]}...")
    print(f"    SA real: {short_answer}")
    print(f"    LA real: {long_answer}")
    print(f"    Amb RAG:  {resp_rag}")
    print(f"    Sense RAG: {resp_sin_rag}")
    print("    ---")

# -----------------------------------
# 4. Guardar tot al DataFrame
# -----------------------------------
qa_df['respuesta_con_rag'] = respuestas_con_rag
qa_df['respuesta_sin_rag'] = respuestas_sin_rag

qa_df['chunk_rec_1_id'] = chunk_rec_1_id
qa_df['chunk_rec_1_doc'] = chunk_rec_1_doc
qa_df['chunk_rec_1_texto'] = chunk_rec_1_texto
qa_df['chunk_rec_1_sim'] = chunk_rec_1_sim

qa_df['chunk_rec_2_id'] = chunk_rec_2_id
qa_df['chunk_rec_2_doc'] = chunk_rec_2_doc
qa_df['chunk_rec_2_texto'] = chunk_rec_2_texto
qa_df['chunk_rec_2_sim'] = chunk_rec_2_sim

qa_df['chunk_rec_3_id'] = chunk_rec_3_id
qa_df['chunk_rec_3_doc'] = chunk_rec_3_doc
qa_df['chunk_rec_3_texto'] = chunk_rec_3_texto
qa_df['chunk_rec_3_sim'] = chunk_rec_3_sim


print("\n✅ Totes les dades guardades al DataFrame!")
print(f"Columnes del DataFrame: {qa_df.columns.tolist()}")

qa_df.to_csv("qa_df_resultados_large.csv", index=False)
files.download("qa_df_resultados_large.csv")

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Models carregats a cpu
Precalculant embeddings dels chunks amb prefix 'passage:'...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

✅ 114 embeddings de chunks calculats

Processant preguntes...
  1/136: What percentage of credits must an undergraduate student pass before starting cu...
    SA real: 50 % of credits.
    LA real: The student must have passed at least 50% of the program credits.
    Amb RAG:  50%
    Sense RAG: a minimum of 80%
    ---
  2/136: Can class schedules overlap with curricular intership schedules?...
    SA real: No.
    LA real: No, class schedules and practicum schedules cannot overlap.
    Amb RAG:  yes
    Sense RAG: yes
    ---
  3/136: What is the minimum number of hours required for a formative project in the curr...
    SA real: 300 hours.
    LA real: The minimum number of hours is 300 hours.
    Amb RAG:  300 hours
    Sense RAG: 10
    ---
  4/136: What is the maximum number of hours required for a formative project in the curr...
    SA real: 375 hours.
    LA real: The maximum number of hours is 375 hours.
    Amb RAG:  300 hours
    Sense RAG: 30
    ---
  5/136: In which peri

KeyboardInterrupt: 

In [ ]:
# tinyllama un par de horas, no me fije
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import torch

# -----------------------------------
# 1. Carregar models
# -----------------------------------
print("Carregant models...")

# Model d'embedding (recuperador) - multilingual-e5-large (requereix prefixos)
modelo_emb = SentenceTransformer('intfloat/multilingual-e5-large')

# Model de llenguatge (generador) - TinyLlama
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer_llm = AutoTokenizer.from_pretrained(model_name)
modelo_llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
device = "cuda" if torch.cuda.is_available() else "cpu"
modelo_llm = modelo_llm.to(device)

# Configurar pad_token
if tokenizer_llm.pad_token is None:
    tokenizer_llm.pad_token = tokenizer_llm.eos_token

print(f"✅ Models carregats a {device}")

# -----------------------------------
# 2. Funció de generació adaptada per a TinyLlama
# -----------------------------------
def generar_respuesta_llm(prompt, max_tokens=150):
    """Genera resposta amb model decoder-only (TinyLlama)."""
    inputs = tokenizer_llm(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = modelo_llm.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=False,
        temperature=None,
        pad_token_id=tokenizer_llm.pad_token_id
    )

    # Decodificar NOMÉS els tokens nous (ignorant el prompt)
    input_len = inputs['input_ids'].shape[1]
    resposta = tokenizer_llm.decode(outputs[0][input_len:], skip_special_tokens=True)
    return resposta.strip()

# -----------------------------------
# 3. Precalcular embeddings dels chunks (amb el prefix "passage: ")
# -----------------------------------
print("Precalculant embeddings dels chunks amb el prefix 'passage:'...")
chunks_textos = chunks_df['text_chunk'].tolist()
chunks_ids = chunks_df['chunk_id'].tolist()
chunks_docs = chunks_df['doc_id'].tolist()

# Afegir el prefix "passage: " a cada chunk (obligatori per a E5)
chunks_con_prefix = ["passage: " + texto for texto in chunks_textos]
chunks_emb = modelo_emb.encode(chunks_con_prefix, show_progress_bar=True)
print(f"✅ {len(chunks_emb)} embeddings de chunks calculats")

# -----------------------------------
# 4. Per a cada pregunta: recuperar chunks (amb prefix "query: ") i generar respostes
# -----------------------------------
top_k = 3
respuestas_con_rag = []
respuestas_sin_rag = []

# Columnes per als chunks
chunk_rec_1_id, chunk_rec_1_doc, chunk_rec_1_texto, chunk_rec_1_sim = [], [], [], []
chunk_rec_2_id, chunk_rec_2_doc, chunk_rec_2_texto, chunk_rec_2_sim = [], [], [], []
chunk_rec_3_id, chunk_rec_3_doc, chunk_rec_3_texto, chunk_rec_3_sim = [], [], [], []

print("\nProcessant preguntes...")
for i, (_, row) in enumerate(qa_df.iterrows()):
    pregunta = row['Q']
    short_answer = row['SA']
    long_answer = row['LA']

    # --- RECUPERACIÓ amb el prefix "query: " ---
    pregunta_con_prefix = "query: " + pregunta
    pregunta_emb = modelo_emb.encode([pregunta_con_prefix], show_progress_bar=False)
    sims = cosine_similarity(pregunta_emb, chunks_emb)[0]
    top_indices = np.argsort(sims)[-top_k:][::-1]

    # Guardar chunks (els textos originals sense prefix)
    for pos, idx in enumerate(top_indices):
        if pos == 0:
            chunk_rec_1_id.append(chunks_ids[idx])
            chunk_rec_1_doc.append(chunks_docs[idx])
            chunk_rec_1_texto.append(chunks_textos[idx])
            chunk_rec_1_sim.append(float(sims[idx]))
        elif pos == 1:
            chunk_rec_2_id.append(chunks_ids[idx])
            chunk_rec_2_doc.append(chunks_docs[idx])
            chunk_rec_2_texto.append(chunks_textos[idx])
            chunk_rec_2_sim.append(float(sims[idx]))
        elif pos == 2:
            chunk_rec_3_id.append(chunks_ids[idx])
            chunk_rec_3_doc.append(chunks_docs[idx])
            chunk_rec_3_texto.append(chunks_textos[idx])
            chunk_rec_3_sim.append(float(sims[idx]))

    # Construir el context amb els textos originals (sense prefix)
    chunks_texto_top = [chunks_textos[idx] for idx in top_indices]
    contexto = "\n\n".join(chunks_texto_top)

    # --- GENERACIÓ AMB RAG ---
    prompt_con_rag = f"""<|system|>
You are a Statistics student at the Faculty of Economics and Business of the University of Barcelona (UB). Use ONLY the following information to answer the question concisely and accurately. If the information is not in the provided documents, say so.</s>
<|user|>
{contexto}

 {pregunta}</s>
<|assistant|>
"""

    resp_rag = generar_respuesta_llm(prompt_con_rag)

    # --- GENERACIÓ SENSE RAG ---
    prompt_sin_rag = f"""<|system|>
You are a Statistics student at the Faculty of Economics and Business of the University of Barcelona (UB). Answer the following question concisely and accurately. If you don't know the answer, say so.</s>
<|user|>
{pregunta}</s>
<|assistant|>
"""

    resp_sin_rag = generar_respuesta_llm(prompt_sin_rag)

    respuestas_con_rag.append(resp_rag)
    respuestas_sin_rag.append(resp_sin_rag)

    print(f"  {i+1}/{len(qa_df)}: {pregunta[:80]}...")
    print(f"    SA real: {short_answer}")
    print(f"    LA real: {long_answer}")
    print(f"    Amb RAG:  {resp_rag}")
    print(f"    Sense RAG: {resp_sin_rag}")
    print("    ---")

# -----------------------------------
# 5. Guardar tot al DataFrame
# -----------------------------------
qa_df['respuesta_con_rag'] = respuestas_con_rag
qa_df['respuesta_sin_rag'] = respuestas_sin_rag

qa_df['chunk_rec_1_id'] = chunk_rec_1_id
qa_df['chunk_rec_1_doc'] = chunk_rec_1_doc
qa_df['chunk_rec_1_texto'] = chunk_rec_1_texto
qa_df['chunk_rec_1_sim'] = chunk_rec_1_sim

qa_df['chunk_rec_2_id'] = chunk_rec_2_id
qa_df['chunk_rec_2_doc'] = chunk_rec_2_doc
qa_df['chunk_rec_2_texto'] = chunk_rec_2_texto
qa_df['chunk_rec_2_sim'] = chunk_rec_2_sim

qa_df['chunk_rec_3_id'] = chunk_rec_3_id
qa_df['chunk_rec_3_doc'] = chunk_rec_3_doc
qa_df['chunk_rec_3_texto'] = chunk_rec_3_texto
qa_df['chunk_rec_3_sim'] = chunk_rec_3_sim

print("\n✅ Totes les dades guardades al DataFrame!")
print(f"Columnes del DataFrame: {qa_df.columns.tolist()}")

qa_df.to_csv("qa_df_resultados_tiny.csv", index=False)
files.download("qa_df_resultados_tiny.csv")

In [ ]:
import numpy as np
import pandas as pd
import torch
import os
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
from huggingface_hub import login
from google.colab import files

# ==============================================
# 2. AUTENTICACIÓN PARA GEMMA
# ==============================================

login(token=os.environ["HF_TOKEN"])

# ==============================================
# 3. CARGAR EMBEDDING (E5) Y CHUNKS
# ==============================================
print("Carregant model d'embedding (E5)...")
modelo_emb = SentenceTransformer('intfloat/multilingual-e5-large')

print("Precalculant embeddings dels chunks amb prefix 'passage:'...")
chunks_textos = chunks_df['text_chunk'].tolist()
chunks_textos_prefijo = ["passage: " + texto for texto in chunks_textos]
chunks_ids = chunks_df['chunk_id'].tolist()
chunks_docs = chunks_df['doc_id'].tolist()
chunks_emb = modelo_emb.encode(chunks_textos_prefijo, show_progress_bar=True)
print(f"✅ {len(chunks_emb)} embeddings de chunks calculats")

# ==============================================
# 4. CARGAR MODEL GENERADOR (GEMMA 2)
# ==============================================
print("Carregant model generador Gemma 2...")
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "google/gemma-2-2b-it"

# Cargar tokenizador
tokenizer_llm = AutoTokenizer.from_pretrained(model_name)

# Detectar dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch_dtype = torch.float16 if device.type == "cuda" else torch.float32

# Cargar modelo (sin device_map)
modelo_llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True
).to(device)

# Configurar pad token
if tokenizer_llm.pad_token is None:
    tokenizer_llm.pad_token = tokenizer_llm.eos_token

# Crear pipeline
pipe = pipeline(
    "text-generation",
    model=modelo_llm,
    tokenizer=tokenizer_llm,
    device=0 if device.type == "cuda" else -1,
    max_new_tokens=150,
    do_sample=False,
    repetition_penalty=1.1
)
print(f"✅ Model generador carregat a {device}")

def generar_respuesta_llm(prompt):
    messages = [{"role": "user", "content": prompt}]
    outputs = pipe(messages, return_full_text=False)
    return outputs[0]['generated_text'].strip()
# ==============================================
# 5. PROCESAR PREGUNTAS (IGUAL QUE TU CÓDIGO ORIGINAL)
# ==============================================
top_k = 3
respuestas_con_rag = []
respuestas_sin_rag = []

chunk_rec_1_id, chunk_rec_1_doc, chunk_rec_1_texto, chunk_rec_1_sim = [], [], [], []
chunk_rec_2_id, chunk_rec_2_doc, chunk_rec_2_texto, chunk_rec_2_sim = [], [], [], []
chunk_rec_3_id, chunk_rec_3_doc, chunk_rec_3_texto, chunk_rec_3_sim = [], [], [], []

print("\nProcessant preguntes...")
for i, (_, row) in enumerate(qa_df.iterrows()):
    pregunta = row['Q']
    short_answer = row['SA']
    long_answer = row['LA']

    # Recuperació amb prefix "query:"
    pregunta_prefijo = "query: " + pregunta
    pregunta_emb = modelo_emb.encode([pregunta_prefijo], show_progress_bar=False)
    sims = cosine_similarity(pregunta_emb, chunks_emb)[0]
    top_indices = np.argsort(sims)[-top_k:][::-1]

    # Guardar chunks recuperats
    for pos, idx in enumerate(top_indices):
        text_chunk = chunks_textos[idx]
        if pos == 0:
            chunk_rec_1_id.append(chunks_ids[idx]); chunk_rec_1_doc.append(chunks_docs[idx])
            chunk_rec_1_texto.append(text_chunk); chunk_rec_1_sim.append(float(sims[idx]))
        elif pos == 1:
            chunk_rec_2_id.append(chunks_ids[idx]); chunk_rec_2_doc.append(chunks_docs[idx])
            chunk_rec_2_texto.append(text_chunk); chunk_rec_2_sim.append(float(sims[idx]))
        elif pos == 2:
            chunk_rec_3_id.append(chunks_ids[idx]); chunk_rec_3_doc.append(chunks_docs[idx])
            chunk_rec_3_texto.append(text_chunk); chunk_rec_3_sim.append(float(sims[idx]))

    # Context per al prompt
    chunks_texto_top = [chunks_textos[idx] for idx in top_indices]
    contexto = "\n\n".join(chunks_texto_top)

    # --- PROMPT AMB RAG ---
    prompt_con_rag = f"""You are a Statistics student at the Faculty of Economics and Business of the University of Barcelona (UB). Use ONLY the following information to answer the question with a complete, natural sentence. If the information is not in the provided documents, say so.

Documents:
{contexto}

Question: {pregunta}
"""
    resp_rag = generar_respuesta_llm(prompt_con_rag)

    # --- PROMPT SENSE RAG ---
    prompt_sin_rag = f"""You are a Statistics student at the Faculty of Economics and Business of the University of Barcelona (UB). Answer the following question with a complete, natural sentence. If you don't know the answer, say so.

Question: {pregunta}
"""
    resp_sin_rag = generar_respuesta_llm(prompt_sin_rag)

    respuestas_con_rag.append(resp_rag)
    respuestas_sin_rag.append(resp_sin_rag)

    print(f"  {i+1}/{len(qa_df)}: {pregunta[:80]}...")
    print(f"    SA real: {short_answer}")
    print(f"    LA real: {long_answer}")
    print(f"    Amb RAG:  {resp_rag}")
    print(f"    Sense RAG: {resp_sin_rag}")
    print("    ---")

# ==============================================
# 6. GUARDAR RESULTATS
# ==============================================
qa_df['respuesta_con_rag'] = respuestas_con_rag
qa_df['respuesta_sin_rag'] = respuestas_sin_rag

qa_df['chunk_rec_1_id'] = chunk_rec_1_id
qa_df['chunk_rec_1_doc'] = chunk_rec_1_doc
qa_df['chunk_rec_1_texto'] = chunk_rec_1_texto
qa_df['chunk_rec_1_sim'] = chunk_rec_1_sim

qa_df['chunk_rec_2_id'] = chunk_rec_2_id
qa_df['chunk_rec_2_doc'] = chunk_rec_2_doc
qa_df['chunk_rec_2_texto'] = chunk_rec_2_texto
qa_df['chunk_rec_2_sim'] = chunk_rec_2_sim

qa_df['chunk_rec_3_id'] = chunk_rec_3_id
qa_df['chunk_rec_3_doc'] = chunk_rec_3_doc
qa_df['chunk_rec_3_texto'] = chunk_rec_3_texto
qa_df['chunk_rec_3_sim'] = chunk_rec_3_sim

print("\n✅ Totes les dades guardades al DataFrame!")
print(f"Columnes del DataFrame: {qa_df.columns.tolist()}")

qa_df.to_csv("qa_df_resultados_gemma2_pipeline.csv", index=False)
files.download("qa_df_resultados_gemma2_pipeline.csv")
print("✅ CSV descarregat: qa_df_resultados_gemma2_pipeline.csv")

Carregant model d'embedding (E5)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Precalculant embeddings dels chunks amb prefix 'passage:'...


...

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  136/136: How is the tuition exemption applied for credits with honors in university studi...
    SA real: Students receive an exemption from tuition fees for a number of credits equivalent to those in which they obtained honors.
    LA real: Students receive an exemption from tuition fees for a number of credits equivalent to those in which they obtained honors.
    Amb RAG:  Students who have achieved an honors distinction in university studies are eligible for an exemption from tuition fees for a certain number of credits equal to those in which they received such distinction.
    Sense RAG: I am not able to provide specific details about how tuition exemptions apply to credits with honors in university studies.  As an AI, I do not have access to real-time information or internal policies of specific universities.
    ---

✅ Totes les dades guardades al DataFrame!
Columnes del DataFrame: ['ID', 'Q', 'SA', 'LA', 'Source_doc', 'Source_text', 'longitud_ref', 'chunk_relevante', 'simili

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ CSV descarregat: qa_df_resultados_gemma2_pipeline.csv


In [ ]:
# ==============================================
# 1. IMPORTACIONES
# ==============================================
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import files

# ==============================================
# 2. CARGAR EMBEDDING (E5) Y CHUNKS
# ==============================================
print("Carregant model d'embedding (E5)...")
modelo_emb = SentenceTransformer('intfloat/multilingual-e5-large')

print("Precalculant embeddings dels chunks amb prefix 'passage:'...")
chunks_textos = chunks_df['text_chunk'].tolist()
chunks_textos_prefijo = ["passage: " + texto for texto in chunks_textos]
chunks_ids = chunks_df['chunk_id'].tolist()
chunks_docs = chunks_df['doc_id'].tolist()
chunks_emb = modelo_emb.encode(chunks_textos_prefijo, show_progress_bar=True)
print(f"✅ {len(chunks_emb)} embeddings de chunks calculats")

# ==============================================
# 3. CARGAR MODEL GENERADOR (QWEN2.5-1.5B) DETERMINISTA
# ==============================================
print("Carregant model generador Qwen2.5-1.5B...")
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer_llm = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
modelo_llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True,
    device_map="auto"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo_llm.to(device)

if tokenizer_llm.pad_token is None:
    tokenizer_llm.pad_token = tokenizer_llm.eos_token

print(f"✅ Model generador Qwen carregat a {device} (mode determinista)")

# ==============================================
# 4. FUNCIÓ DE GENERACIÓ (determinista)
# ==============================================
def generar_respuesta_llm(prompt, max_tokens=150):
    # Qwen espera un chat template
    messages = [{"role": "user", "content": prompt}]
    formatted_prompt = tokenizer_llm.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer_llm(formatted_prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = modelo_llm.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=False,                # determinista
        pad_token_id=tokenizer_llm.pad_token_id,
        eos_token_id=tokenizer_llm.eos_token_id
    )
    input_len = inputs['input_ids'].shape[1]
    resposta = tokenizer_llm.decode(outputs[0][input_len:], skip_special_tokens=True)
    return resposta.strip()

# ==============================================
# 5. PROCESSAR PREGUNTES
# ==============================================
top_k = 3
respuestas_con_rag = []
respuestas_sin_rag = []

chunk_rec_1_id, chunk_rec_1_doc, chunk_rec_1_texto, chunk_rec_1_sim = [], [], [], []
chunk_rec_2_id, chunk_rec_2_doc, chunk_rec_2_texto, chunk_rec_2_sim = [], [], [], []
chunk_rec_3_id, chunk_rec_3_doc, chunk_rec_3_texto, chunk_rec_3_sim = [], [], [], []

print("\nProcessant preguntes...")
for i, (_, row) in enumerate(qa_df.iterrows()):
    pregunta = row['Q']
    short_answer = row['SA']
    long_answer = row['LA']

    # Recuperació amb prefix "query:"
    pregunta_prefijo = "query: " + pregunta
    pregunta_emb = modelo_emb.encode([pregunta_prefijo], show_progress_bar=False)
    sims = cosine_similarity(pregunta_emb, chunks_emb)[0]
    top_indices = np.argsort(sims)[-top_k:][::-1]

    for pos, idx in enumerate(top_indices):
        text_chunk = chunks_textos[idx]
        if pos == 0:
            chunk_rec_1_id.append(chunks_ids[idx]); chunk_rec_1_doc.append(chunks_docs[idx])
            chunk_rec_1_texto.append(text_chunk); chunk_rec_1_sim.append(float(sims[idx]))
        elif pos == 1:
            chunk_rec_2_id.append(chunks_ids[idx]); chunk_rec_2_doc.append(chunks_docs[idx])
            chunk_rec_2_texto.append(text_chunk); chunk_rec_2_sim.append(float(sims[idx]))
        elif pos == 2:
            chunk_rec_3_id.append(chunks_ids[idx]); chunk_rec_3_doc.append(chunks_docs[idx])
            chunk_rec_3_texto.append(text_chunk); chunk_rec_3_sim.append(float(sims[idx]))

    chunks_texto_top = [chunks_textos[idx] for idx in top_indices]
    contexto = "\n\n".join(chunks_texto_top)

    # --- PROMPT AMB RAG (natural, sense etiquetes especials, Qwen les aplicarà internament) ---
    prompt_con_rag = f"""You are a Statistics student at the Faculty of Economics and Business of the University of Barcelona (UB). Use ONLY the following information to answer the question with a complete, natural sentence. If the information is not in the provided documents, say so.

Documents:
{contexto}

Question: {pregunta}
"""
    resp_rag = generar_respuesta_llm(prompt_con_rag)

    # --- PROMPT SENSE RAG ---
    prompt_sin_rag = f"""You are a Statistics student at the Faculty of Economics and Business of the University of Barcelona (UB). Answer the following question with a complete, natural sentence. If you don't know the answer, say so.

Question: {pregunta}
"""
    resp_sin_rag = generar_respuesta_llm(prompt_sin_rag)

    respuestas_con_rag.append(resp_rag)
    respuestas_sin_rag.append(resp_sin_rag)

    print(f"  {i+1}/{len(qa_df)}: {pregunta[:80]}...")
    print(f"    SA real: {short_answer}")
    print(f"    LA real: {long_answer}")
    print(f"    Amb RAG:  {resp_rag}")
    print(f"    Sense RAG: {resp_sin_rag}")
    print("    ---")

# ==============================================
# 6. GUARDAR RESULTATS
# ==============================================
qa_df['respuesta_con_rag'] = respuestas_con_rag
qa_df['respuesta_sin_rag'] = respuestas_sin_rag

qa_df['chunk_rec_1_id'] = chunk_rec_1_id
qa_df['chunk_rec_1_doc'] = chunk_rec_1_doc
qa_df['chunk_rec_1_texto'] = chunk_rec_1_texto
qa_df['chunk_rec_1_sim'] = chunk_rec_1_sim

qa_df['chunk_rec_2_id'] = chunk_rec_2_id
qa_df['chunk_rec_2_doc'] = chunk_rec_2_doc
qa_df['chunk_rec_2_texto'] = chunk_rec_2_texto
qa_df['chunk_rec_2_sim'] = chunk_rec_2_sim

qa_df['chunk_rec_3_id'] = chunk_rec_3_id
qa_df['chunk_rec_3_doc'] = chunk_rec_3_doc
qa_df['chunk_rec_3_texto'] = chunk_rec_3_texto
qa_df['chunk_rec_3_sim'] = chunk_rec_3_sim

print("\n✅ Totes les dades guardades al DataFrame!")
print(f"Columnes del DataFrame: {qa_df.columns.tolist()}")

qa_df.to_csv("qa_df_resultados_qwen15b_determinista.csv", index=False)
files.download("qa_df_resultados_qwen15b_determinista.csv")
print("✅ CSV descarregat: qa_df_resultados_qwen15b_determinista.csv")

Carregant model d'embedding (E5)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Precalculant embeddings dels chunks amb prefix 'passage:'...


...

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  1/136: What percentage of credits must an undergraduate student pass before starting cu...
    SA real: 50 % of credits.
    LA real: The student must have passed at least 50% of the program credits.
    Amb RAG:  An undergraduate student must have passed 50% of the credits for the program before starting curricular internships.
    Sense RAG: According to university regulations, typically 75% or more of the required credits must be passed for an undergraduate student to begin curricular internships.
    ---
  2/136: Can class schedules overlap with curricular intership schedules?...
    SA real: No.
    LA real: No, class schedules and practicum schedules cannot overlap.
    Amb RAG:  No, class schedules cannot overlap with curricular internship schedules; interns must be available during their scheduled working hours.
    Sense RAG: Yes, it is possible for class schedules to overlap with curricular intership schedules, as students may have multiple commitments during their academic

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ CSV descarregat: qa_df_resultados_qwen15b_determinista.csv
